In [2]:
import pandas as pd

In [5]:
vt = pd.read_csv("../data/clean_data/voting_turnout_clean.csv.zip", dtype={"GDENR": str}, na_values="...", low_memory=False)
#dtype={"GDENR": str}: signlas read the column GDENR as text. The column is empty for Switzerland, cantons and districts. Without this, pandas would turn the empty cells into NaN and store all numbers as decimals (261.0 instead of 261).
#na_values="...": in this file, a missing value is written as three dots .... This tells pandas: „treat ... as missing (NaN)
#low_memory=False: the file is very big, so by default pandas reads it in pieces and guesses the column types separately for each piece. That causes a warning („mixed types“). With False, pandas reads everything at once and guesses consistently.
print(vt.shape)
vt.head()
vt.sample(20)

(1207493, 9)


,Kanton (-) / Bezirk (>>) / Gemeinde (......),Datum und Vorlage,Stimmberechtigte,Beteiligung in %,Geografietyp,Name,Sonderfall,GDENR,Kanton
10181,......Berg am Irchel,1972-06-04 Bundesbeschluss über den Schutz der...,246.0,63.41,Gemeinde,Berg am Irchel,False,23,Zürich
174894,......Rüdtligen-Alchenflüh,2012-03-11 Volksinitiative «6 Wochen Ferien fü...,1439.0,29.95,Gemeinde,Rüdtligen-Alchenflüh,False,420,Bern / Berne
115939,......Bellmund,1975-06-08 Bundesbeschluss über den Schutz der...,460.0,40.87,Gemeinde,Bellmund,False,732,Bern / Berne
1179356,......Boécourt,1971-02-07 Bundesbeschluss über die Einführung...,205.0,56.59,Gemeinde,Boécourt,False,6702,Jura
632235,......Oberbüren,2012-06-17 Volksinitiative «Für die Stärkung d...,2839.0,29.62,Gemeinde,Oberbüren,False,3424,St. Gallen
888119,......Tresa,2026-06-14 Volksinitiative «Keine 10-Millionen...,2275.0,50.86,Gemeinde,Tresa,False,5239,Ticino
739812,......Leimbach (AG),1980-11-30 Bundesbeschluss über die Aufhebung ...,NaN,NaN,Gemeinde,Leimbach (AG),False,4137,Aargau
31391,>> Bezirk Dielsdorf,2000-11-26 Bundespersonalgesetz,42729.0,36.68,Bezirk,Bezirk Dielsdorf,False,NaN,Zürich
25188,......Hüntwangen,2009-05-17 Bundesbeschluss über die Genehmigun...,698.0,45.42,Gemeinde,Hüntwangen,False,61,Zürich
1053540,......Crissier,1985-03-10 Bundesbeschluss über die Aufhebung ...,2425.0,24.37,Gemeinde,Crissier,False,5583,Vaud


In [6]:
df = vt[vt["Geografietyp"] == "Gemeinde"] 
#only keep gemeinden
df = df[df['Kanton'] == 'Zürich']
# only canton Zurich
df = df[df["Sonderfall"] == False] 
# no special cases (e.g. Swiss abroad)
print(df.shape)
print(df["GDENR"].nunique())

(81760, 9)
160


In [7]:
df = df.copy()
#such that new table is built

In [8]:
df["year"] = df["Datum und Vorlage"].str[:4]
df["year"].head()

1533    2026
1534    2026
1535    2026
1536    2026
1537    2026
Name: year, dtype: str

In [9]:
df["year"] = df["year"].astype(int)
#turn year text into ints   

In [10]:
df["GDENR"] = df["GDENR"].astype(int)

In [11]:
print(df.dtypes)
print(df["year"].min(), df["year"].max())

Kanton (-) / Bezirk (>>) / Gemeinde (......)        str
Datum und Vorlage                                   str
Stimmberechtigte                                float64
Beteiligung in %                                float64
Geografietyp                                        str
Name                                                str
Sonderfall                                         bool
GDENR                                             int64
Kanton                                              str
year                                              int64
dtype: object
1960 2026


In [12]:
df = df[["GDENR", "year", "Beteiligung in %"]]

df = df.rename(columns={"Beteiligung in %": "turnout"})

df.head()

,GDENR,year,turnout
1533,1,2026,71.98
1534,1,2026,73.64
1535,1,2026,69.09
1536,1,2026,69.52
1537,1,2026,69.60


In [13]:
print(df.dtypes)
print(df.isna().sum())

GDENR        int64
year         int64
turnout    float64
dtype: object
GDENR      0
year       0
turnout    0
dtype: int64


In [ ]:
turnout = df.groupby(["GDENR", "year"])["turnout"].mean()


In [15]:
turnout.head()


GDENR  year
1      1960    53.425000
       1961    58.072500
       1962    61.112500
       1963    52.613333
       1964    52.213333
Name: turnout, dtype: float64

In [ ]:
turnout = turnout.reset_index()
#make columns normal again

In [17]:
turnout.head()

,GDENR,year,turnout
0,1,1960,53.425000
1,1,1961,58.072500
2,1,1962,61.112500
3,1,1963,52.613333
4,1,1964,52.213333


In [18]:
print(turnout.duplicated(subset=["GDENR", "year"]).sum())
print(turnout["GDENR"].nunique())
print(turnout["turnout"].describe())
print(turnout.groupby("year")["GDENR"].nunique())

0
160
count    10720.000000
mean        52.012220
std         10.084242
min         24.613333
25%         44.913409
50%         51.237532
75%         58.230625
max         91.723333
Name: turnout, dtype: float64
year
1960    160
1961    160
1962    160
1963    160
1964    160
       ... 
2022    160
2023    160
2024    160
2025    160
2026    160
Name: GDENR, Length: 67, dtype: int64


In [19]:
turnout.to_csv("../data/clean_data/ts_zh_turnout.csv", index=False)